# Full-image reconstruction from saved checkpoint

Load a trained SimMIM (+VICReg) encoder+decoder from `outputs/` and run
sliding-window reconstruction on full microscopy images.
No training — visualisation only.

## Imports

In [ ]:
import os, sys, json
from pathlib import Path

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

## Path setup

In [ ]:
NB_DIR = Path.cwd().resolve()
REPO_ROOT = NB_DIR
while REPO_ROOT.parent != REPO_ROOT and not (REPO_ROOT / '.git').is_dir():
    REPO_ROOT = REPO_ROOT.parent
ROOT = REPO_ROOT / 'src'
for p in (ROOT, REPO_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
print('REPO_ROOT:', REPO_ROOT)
print('ROOT     :', ROOT)

In [ ]:
from synaptic_ssl.training.config import ModelCfg
from synaptic_ssl.training.config import SSLCfg
from synaptic_ssl.training.seeding import seed_everything
from synaptic_ssl.training.masking import random_block_mask, apply_mask
from synaptic_ssl.training.checkpoints import load_checkpoint
from synaptic_ssl.models.swin import build_swin_encoder, build_simmim_vicreg_heads, count_params
from synaptic_ssl.ssl_training.post_training import build_run_label


## Configuration

Point `CHECKPOINT_PATH` to a training run directory that contains
`best_model.pt` and `config.json`, **or** directly to a `.pt` file.
`DATA_ROOT` must point at the patched dataset so we can reassemble
full images.

In [ ]:
CHECKPOINT_PATH = REPO_ROOT / 'outputs' / 'simmim_vicreg_pretrain_XXXXXXXX_XXXXXX'  # <-- edit this
DATA_ROOT       = REPO_ROOT / 'data' / 'patches_128'
EXCLUDE         = ["KONTROLA"]
CHANNEL_NAMES   = ["pre_synaptic", "post_synaptic", "structural"]

FULL_IMAGE_INDEX = None   # None -> first non-excluded image
OVERLAP          = 0.5    # sliding-window overlap fraction
BATCH_SIZE       = 16
SEED             = 42

## Resolve checkpoint

In [ ]:
ckpt_path = Path(CHECKPOINT_PATH)
if ckpt_path.is_dir():
    best = ckpt_path / "best_model.pt"
    if not best.exists():
        best = ckpt_path / "last.pt"
    ckpt_path = best
assert ckpt_path.exists(), f"Checkpoint not found: {ckpt_path}"

run_dir = ckpt_path.parent
print(f"checkpoint : {ckpt_path}")
print(f"run dir    : {run_dir}")

In [ ]:
# Load config.json from the run directory to recover
# model / ssl / channel-stats settings automatically.
cfg_path = run_dir / "config.json"
if cfg_path.exists():
    with open(cfg_path) as f:
        run_cfg = json.load(f)
    print("loaded config.json")
else:
    run_cfg = {}
    print("no config.json found — using defaults")

In [ ]:
# Channel stats: prefer the run's own channel_stats.json.
stats_path = run_dir / "channel_stats.json"
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    ch_mean = torch.tensor(stats["mean"], dtype=torch.float32)
    ch_std  = torch.tensor(stats["std"],  dtype=torch.float32)
    print(f"channel stats from {stats_path}")
else:
    # Fallback: hard-coded from the moby training run.
    ch_mean = torch.tensor([0.10903, 0.11901, 0.15514])
    ch_std  = torch.tensor([0.12975, 0.13056, 0.14075])
    print("using hard-coded channel stats")
print(f"  mean = {ch_mean.tolist()}")
print(f"  std  = {ch_std.tolist()}")

## Build model and load weights

In [ ]:
# Reconstruct the same ModelCfg and SSLCfg used during training.
_mcfg = run_cfg.get("model", {})
model_cfg = ModelCfg(
    in_channels       = _mcfg.get("in_channels", 3),
    img_size          = _mcfg.get("img_size", 128),
    feature_size      = _mcfg.get("feature_size", 96),
    patch_size        = _mcfg.get("patch_size", 2),
    window_size       = _mcfg.get("window_size", 7),
    depths            = tuple(_mcfg.get("depths", (2, 2, 6, 2))),
    num_heads         = tuple(_mcfg.get("num_heads", (3, 6, 12, 24))),
    dropout_path_rate = _mcfg.get("dropout_path_rate", 0.05),
)

_scfg = run_cfg.get("ssl", {})
ssl_cfg = SSLCfg(
    mask_ratio       = _scfg.get("mask_ratio", 0.40),
    mask_block_size  = _scfg.get("mask_block_size", 16),
    loss_kind        = _scfg.get("loss_kind", "l1"),
    lambda_sim       = _scfg.get("lambda_sim", 25.0),
    lambda_std       = _scfg.get("lambda_std", 25.0),
    lambda_cov       = _scfg.get("lambda_cov", 1.0),
    w_recon          = _scfg.get("w_recon", 1.0),
    w_vicreg         = _scfg.get("w_vicreg", 0.0),
    projector_hidden = _scfg.get("projector_hidden", 512),
    projector_dim    = _scfg.get("projector_dim", 512),
    fg_weight_alpha  = _scfg.get("fg_weight_alpha", 5.0),
    fg_weight_tau    = _scfg.get("fg_weight_tau", 1.0),
    fg_weight_temp   = _scfg.get("fg_weight_temp", 0.5),
)
print(model_cfg)
print(ssl_cfg)

In [ ]:
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
if device.type == "cuda":
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"gpu    = {name}  ({mem:.1f} GB)")

In [ ]:
encoder = build_swin_encoder(model_cfg).to(device)
heads   = build_simmim_vicreg_heads(model_cfg, ssl_cfg)
heads   = {k: v.to(device) for k, v in heads.items()}
print(f"encoder params = {count_params(encoder) / 1e6:.2f} M")
print(f"head params    = {count_params(heads) / 1e6:.2f} M")

In [ ]:
ckpt = load_checkpoint(
    ckpt_path, encoder=encoder, heads=heads, map_location=device,
)
print(f"loaded checkpoint: epoch={ckpt.get('epoch')}  val_metric={ckpt.get('val_metric')}")

run_label = build_run_label(
    method_name=run_cfg.get("base", {}).get("method_name", "simmim_vicreg"),
    init_source=run_cfg.get("base", {}).get("init_source", "moby"),
    epoch=ckpt.get("epoch"),
    val_metric=ckpt.get("val_metric"),
)

## Reassemble full image

In [ ]:
from synaptic_ssl.utils_data.reassemble import reassemble_image, list_image_indices

idxs = list_image_indices(DATA_ROOT, exclude_patterns=EXCLUDE)
img_idx = idxs[0] if FULL_IMAGE_INDEX is None else FULL_IMAGE_INDEX
full_image, _records = reassemble_image(
    DATA_ROOT, img_idx, exclude_patterns=EXCLUDE,
)
print(f"image idx={img_idx}  shape={tuple(full_image.shape)}  "
      f'source={_records[0]["source_image"]}')

## True full reconstruction (no masking)

Feed every sliding window through the encoder and decoder **without**
masking. This shows what the trained decoder actually reconstructs
from the encoder features — the "true" reconstruction ability of the
model on full, unmasked input.

In [ ]:
@torch.no_grad()
def full_image_true_recon(
    encoder, heads, full_image, *,
    patch_size, head_stage_index,
    ch_mean, ch_std, device,
    overlap=0.5, batch_size=16,
):
    """Sliding-window reconstruction WITHOUT masking.

    Every window is z-scored, encoded, decoded, de-z-scored, and
    accumulated.  Overlapping regions are averaged.
    """
    encoder.eval()
    for h in heads.values():
        h.eval()

    C, H, W = full_image.shape
    stride = max(1, int(patch_size * (1.0 - overlap)))

    ch_mean_np = np.asarray(ch_mean.cpu(), dtype=np.float32).reshape(-1, 1, 1)
    ch_std_np  = np.asarray(ch_std.cpu(),  dtype=np.float32).reshape(-1, 1, 1)

    pad_h = (stride - (H - patch_size) % stride) % stride if H > patch_size else patch_size - H
    pad_w = (stride - (W - patch_size) % stride) % stride if W > patch_size else patch_size - W
    padded = np.pad(full_image, ((0, 0), (0, pad_h), (0, pad_w)), mode="reflect")
    pH, pW = padded.shape[1], padded.shape[2]

    acc_recon = np.zeros((C, pH, pW), dtype=np.float64)
    count     = np.zeros((pH, pW),    dtype=np.float64)

    positions = [
        (y, x)
        for y in range(0, pH - patch_size + 1, stride)
        for x in range(0, pW - patch_size + 1, stride)
    ]

    mu = ch_mean.view(1, -1, 1, 1).to(device)
    sd = ch_std.view(1, -1, 1, 1).to(device)

    for i in tqdm(range(0, len(positions), batch_size), desc="true recon"):
        batch_pos = positions[i : i + batch_size]
        patches = [
            (padded[:, y : y + patch_size, x : x + patch_size] - ch_mean_np) / ch_std_np
            for y, x in batch_pos
        ]
        view = torch.from_numpy(np.stack(patches)).float().to(device)

        z = encoder(view.contiguous())[head_stage_index]
        recon = heads["decoder"](z).float()
        recon01 = (recon * sd + mu).clamp(0, 1).cpu().numpy()

        for j, (y, x) in enumerate(batch_pos):
            acc_recon[:, y : y + patch_size, x : x + patch_size] += recon01[j]
            count[y : y + patch_size, x : x + patch_size] += 1.0

    cs = np.maximum(count, 1.0)
    full_recon = (acc_recon / cs)[:, :H, :W].astype(np.float32)
    return full_recon, len(positions)

In [ ]:
full_recon, n_windows = full_image_true_recon(
    encoder, heads, full_image,
    patch_size=model_cfg.img_size,
    head_stage_index=ssl_cfg.head_stage_index,
    ch_mean=ch_mean, ch_std=ch_std,
    device=device,
    overlap=OVERLAP,
    batch_size=BATCH_SIZE,
)

err = np.abs(full_recon - full_image).mean(0)
print(
    f"true full-image recon: windows={n_windows}  "
    f"stride={int(model_cfg.img_size * (1 - OVERLAP))}px  "
    f"mean|err|={float(err.mean()):.4f}  max|err|={float(err.max()):.4f}"
)

## Visualisation — true reconstruction

In [ ]:
C, H, W = full_image.shape
ncols = max(3, C)
fig, axes = plt.subplots(4, ncols, figsize=(5 * ncols, 20), squeeze=False)
for r in range(4):
    for c in range(ncols):
        axes[r, c].axis("off")

for ci, name in enumerate(CHANNEL_NAMES):
    axes[0, ci].imshow(full_image[ci], cmap="magma", vmin=0, vmax=1)
    axes[0, ci].set_title(f"target  {name}")
    axes[1, ci].imshow(full_recon[ci], cmap="magma", vmin=0, vmax=1)
    axes[1, ci].set_title(f"reconstruction  {name}")
    ch_err = np.abs(full_recon[ci] - full_image[ci])
    ch_err_vmax = max(float(ch_err.max()), 1e-6)
    im = axes[2, ci].imshow(ch_err, cmap="hot", vmin=0, vmax=ch_err_vmax)
    axes[2, ci].set_title(f"|err|  {name}  mean={float(ch_err.mean()):.3f}")
    fig.colorbar(im, ax=axes[2, ci], fraction=0.046, pad=0.02)

# Row 4: composite RGB
axes[3, 0].imshow(np.transpose(np.clip(full_image, 0, 1), (1, 2, 0)))
axes[3, 0].set_title("target  RGB")
axes[3, 1].imshow(np.transpose(np.clip(full_recon, 0, 1), (1, 2, 0)))
axes[3, 1].set_title("reconstruction  RGB")
err_vmax = max(float(err.max()), 1e-6)
err_im = axes[3, 2].imshow(err, cmap="hot", vmin=0, vmax=err_vmax)
axes[3, 2].set_title(f"|err| mean  mean={float(err.mean()):.3f}")
fig.colorbar(err_im, ax=axes[3, 2], fraction=0.046, pad=0.02)

fig.suptitle(
    f"True full-image reconstruction (no masking)\n"
    f"img={img_idx}, stride={int(model_cfg.img_size * (1 - OVERLAP))}px, "
    f"windows={n_windows}",
    fontsize=14,
)
if run_label:
    fig.text(0.01, 0.985, run_label, ha="left", va="top", fontsize=8,
             color="#444", family="DejaVu Sans Mono")
fig.tight_layout(rect=(0, 0, 0.97, 0.95))
plt.show()

## Masked reconstruction (same as training)

For comparison, also run the sliding-window reconstruction **with**
block masking (same recipe as training). Differences between this
and the true reconstruction above reveal how much the decoder
relies on visible-patch context vs. learned features.

In [ ]:
@torch.no_grad()
def full_image_masked_recon(
    encoder, heads, full_image, *,
    patch_size, mask_block_size, mask_ratio, head_stage_index,
    ch_mean, ch_std, device,
    overlap=0.5, batch_size=16, seed=0,
):
    """Sliding-window SimMIM reconstruction WITH masking."""
    encoder.eval()
    for h in heads.values():
        h.eval()

    C, H, W = full_image.shape
    stride = max(1, int(patch_size * (1.0 - overlap)))

    ch_mean_np = np.asarray(ch_mean.cpu(), dtype=np.float32).reshape(-1, 1, 1)
    ch_std_np  = np.asarray(ch_std.cpu(),  dtype=np.float32).reshape(-1, 1, 1)

    pad_h = (stride - (H - patch_size) % stride) % stride if H > patch_size else patch_size - H
    pad_w = (stride - (W - patch_size) % stride) % stride if W > patch_size else patch_size - W
    padded = np.pad(full_image, ((0, 0), (0, pad_h), (0, pad_w)), mode="reflect")
    pH, pW = padded.shape[1], padded.shape[2]

    acc_recon  = np.zeros((C, pH, pW), dtype=np.float64)
    acc_masked = np.zeros((C, pH, pW), dtype=np.float64)
    acc_mask   = np.zeros((pH, pW),    dtype=np.float64)
    count      = np.zeros((pH, pW),    dtype=np.float64)

    positions = [
        (y, x)
        for y in range(0, pH - patch_size + 1, stride)
        for x in range(0, pW - patch_size + 1, stride)
    ]

    torch.manual_seed(seed)
    mu = ch_mean.view(1, -1, 1, 1).to(device)
    sd = ch_std.view(1, -1, 1, 1).to(device)

    for i in tqdm(range(0, len(positions), batch_size), desc="masked recon"):
        batch_pos = positions[i : i + batch_size]
        patches = [
            (padded[:, y : y + patch_size, x : x + patch_size] - ch_mean_np) / ch_std_np
            for y, x in batch_pos
        ]
        view = torch.from_numpy(np.stack(patches)).float().to(device)

        mask = random_block_mask(view, mask_block_size, mask_ratio)
        v_masked = apply_mask(view, mask, heads["mask_token"])
        z = encoder(v_masked.contiguous())[head_stage_index]
        recon = heads["decoder"](z).float()

        recon01  = (recon    * sd + mu).clamp(0, 1).cpu().numpy()
        masked01 = (v_masked * sd + mu).clamp(0, 1).cpu().numpy()
        mask_np  = mask.cpu().numpy()[:, 0]

        for j, (y, x) in enumerate(batch_pos):
            acc_recon [:, y : y + patch_size, x : x + patch_size] += recon01[j]
            acc_masked[:, y : y + patch_size, x : x + patch_size] += masked01[j]
            acc_mask  [   y : y + patch_size, x : x + patch_size] += mask_np[j]
            count     [   y : y + patch_size, x : x + patch_size] += 1.0

    cs = np.maximum(count, 1.0)
    full_recon  = (acc_recon  / cs)[:, :H, :W].astype(np.float32)
    full_masked = (acc_masked / cs)[:, :H, :W].astype(np.float32)
    mask_freq   = (acc_mask   / cs)[:H, :W].astype(np.float32)
    return full_recon, full_masked, mask_freq, len(positions)

In [ ]:
masked_recon, masked_input, mask_freq, n_win_m = full_image_masked_recon(
    encoder, heads, full_image,
    patch_size=model_cfg.img_size,
    mask_block_size=ssl_cfg.mask_block_size,
    mask_ratio=ssl_cfg.mask_ratio,
    head_stage_index=ssl_cfg.head_stage_index,
    ch_mean=ch_mean, ch_std=ch_std,
    device=device,
    overlap=OVERLAP,
    batch_size=BATCH_SIZE,
    seed=SEED + 31,
)

err_m = np.abs(masked_recon - full_image).mean(0)
mask_pct = int(ssl_cfg.mask_ratio * 100)
print(
    f"masked full-image recon: windows={n_win_m}  "
    f"mask={mask_pct}% blk={ssl_cfg.mask_block_size}  "
    f"mean|err|={float(err_m.mean()):.4f}  max|err|={float(err_m.max()):.4f}"
)

## Visualisation — masked reconstruction

In [ ]:
C, H, W = full_image.shape
ncols = max(3, C)
fig, axes = plt.subplots(4, ncols, figsize=(5 * ncols, 20), squeeze=False)
for r in range(4):
    for c in range(ncols):
        axes[r, c].axis("off")

for ci, name in enumerate(CHANNEL_NAMES):
    axes[0, ci].imshow(full_image[ci], cmap="magma", vmin=0, vmax=1)
    axes[0, ci].set_title(f"target  {name}")
    axes[1, ci].imshow(masked_input[ci], cmap="magma", vmin=0, vmax=1)
    axes[1, ci].set_title(f"masked input (avg)  {name}")
    axes[2, ci].imshow(masked_recon[ci], cmap="magma", vmin=0, vmax=1)
    axes[2, ci].set_title(f"reconstruction  {name}")

axes[3, 0].imshow(np.transpose(np.clip(full_image, 0, 1), (1, 2, 0)))
axes[3, 0].set_title("target  RGB")
axes[3, 1].imshow(np.transpose(np.clip(masked_recon, 0, 1), (1, 2, 0)))
axes[3, 1].set_title("reconstruction  RGB")
err_m_vmax = max(float(err_m.max()), 1e-6)
err_m_im = axes[3, 2].imshow(err_m, cmap="hot", vmin=0, vmax=err_m_vmax)
axes[3, 2].set_title(f"|err|  mean={float(err_m.mean()):.3f}")
fig.colorbar(err_m_im, ax=axes[3, 2], fraction=0.046, pad=0.02)

fig.suptitle(
    f"Masked sliding-window reconstruction\n"
    f"img={img_idx}, stride={int(model_cfg.img_size * (1 - OVERLAP))}px, "
    f"mask={mask_pct}% blk={ssl_cfg.mask_block_size}, "
    f"windows={n_win_m}",
    fontsize=14,
)
if run_label:
    fig.text(0.01, 0.985, run_label, ha="left", va="top", fontsize=8,
             color="#444", family="DejaVu Sans Mono")
fig.tight_layout(rect=(0, 0, 0.97, 0.95))
plt.show()

## Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax in axes.flat:
    ax.axis("off")

axes[0, 0].imshow(np.transpose(np.clip(full_image, 0, 1), (1, 2, 0)))
axes[0, 0].set_title("Ground truth", fontsize=13)
axes[0, 1].imshow(np.transpose(np.clip(full_recon, 0, 1), (1, 2, 0)))
axes[0, 1].set_title(f"True recon (unmasked)  |err|={float(err.mean()):.4f}", fontsize=13)
axes[0, 2].imshow(np.transpose(np.clip(masked_recon, 0, 1), (1, 2, 0)))
axes[0, 2].set_title(f"Masked recon ({mask_pct}%)  |err|={float(err_m.mean()):.4f}", fontsize=13)

# Error maps (shared colour scale)
err_true   = np.abs(full_recon - full_image).mean(0)
err_masked = np.abs(masked_recon - full_image).mean(0)
vmax = max(float(err_true.max()), float(err_masked.max()), 1e-6)

im1 = axes[1, 0].imshow(err_true, cmap="hot", vmin=0, vmax=vmax)
axes[1, 0].set_title(f"Error (true recon)  mean={float(err_true.mean()):.4f}")
fig.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.02)

im2 = axes[1, 1].imshow(err_masked, cmap="hot", vmin=0, vmax=vmax)
axes[1, 1].set_title(f"Error (masked recon)  mean={float(err_masked.mean()):.4f}")
fig.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.02)

diff = err_masked - err_true
diff_abs = max(abs(float(diff.min())), abs(float(diff.max())), 1e-6)
im3 = axes[1, 2].imshow(diff, cmap="RdBu_r", vmin=-diff_abs, vmax=diff_abs)
axes[1, 2].set_title("Masked err − True err")
fig.colorbar(im3, ax=axes[1, 2], fraction=0.046, pad=0.02)

fig.suptitle("Side-by-side: true vs masked reconstruction", fontsize=15)
if run_label:
    fig.text(0.01, 0.985, run_label, ha="left", va="top", fontsize=8,
             color="#444", family="DejaVu Sans Mono")
fig.tight_layout(rect=(0, 0, 0.97, 0.95))
plt.show()

## Per-channel SSIM (optional)

In [ ]:
try:
    from skimage.metrics import structural_similarity as ssim
    print("Per-channel SSIM (true recon vs ground truth):")
    for ci, name in enumerate(CHANNEL_NAMES):
        s = ssim(full_image[ci], full_recon[ci], data_range=1.0)
        print(f"  {name:>14s}:  SSIM = {s:.4f}")
    print()
    print("Per-channel SSIM (masked recon vs ground truth):")
    for ci, name in enumerate(CHANNEL_NAMES):
        s = ssim(full_image[ci], masked_recon[ci], data_range=1.0)
        print(f"  {name:>14s}:  SSIM = {s:.4f}")
except ImportError:
    print("skimage not available — skipping SSIM")